<div style="background: linear-gradient(135deg, #0f0f0f 0%, #1a1a2e 50%, #16213e 100%); border-radius: 16px; padding: 40px 30px; text-align: center; font-family: 'Segoe UI', sans-serif; border: 1px solid #333;">
  <h1 style="color: #fff; font-size: 2.4em; margin: 0 0 8px 0; letter-spacing: 2px;">🎙️ Yogeshwar AI</h1>
  <h2 style="color: #a78bfa; font-size: 1.3em; font-weight: 400; margin: 0 0 20px 0;">RVC Voice Cloner — Train & Inference</h2>
  <p style="color: #888; font-size: 0.95em; margin: 0 0 20px 0;">Powered by RVC v2 · GPU Accelerated · Python 3.12 Compatible</p>
  <div style="display: flex; justify-content: center; gap: 16px; flex-wrap: wrap;">
    <a href='https://www.instagram.com/theyogeshwar' target='_blank' style="background:#833ab4; color:#fff; padding:8px 20px; border-radius:8px; text-decoration:none; font-size:0.9em;">📸 @theyogeshwar</a>
  </div>
  <p style="color: #555; font-size: 0.78em; margin: 20px 0 0 0;">⚠️ Use responsibly. Only clone voices you have permission for.</p>
</div>

---
**Last Update : March 2026** · [Instagram @theyogeshwar](https://www.instagram.com/theyogeshwar)

---

### ⚡ Quick Start Order
Run cells **in order**: Step 1 → Step 2 → Step 3 → Step 4

> ⚠️ **If your runtime restarts**, you must re-run from Step 1. Your uploaded audio in `/content/dataset` usually survives — Step 2 will auto-detect it.

---
## 🔧 Setup

In [ ]:
#@title ⚙️ **Step 1 : Install RVC** (Python 3.12 + CUDA 12.1)
#@markdown Installs all dependencies. Run once per session. Takes ~5 minutes.

import os, subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:] if result.stdout else "")
        print(result.stderr[-3000:] if result.stderr else "")
        raise RuntimeError(f"Failed: {cmd}")
    return result

# ── 1. Clone RVC ─────────────────────────────────────────────
if not os.path.exists('/content/RVC'):
    print("📦 Cloning RVC repository...")
    run("git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI /content/RVC")
else:
    print("✅ RVC folder already exists.")

# ── 2. Install uv ────────────────────────────────────────────
print("🔧 Installing uv...")
run("pip install uv -q")

# ── 3. PyTorch 2.4.1 — first version with cp312 wheels ───────
print("🔧 Installing PyTorch 2.4.1 (cp312 + CUDA 12.1)...")
run("uv pip install --system "
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 "
    "--index-url https://download.pytorch.org/whl/cu121")

# ── 4. Verify torch ───────────────────────────────────────────
r = subprocess.run([sys.executable, "-c",
    "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available())"],
    capture_output=True, text=True)
print("  ", r.stdout.strip())

# ── 5. torchcrepe ────────────────────────────────────────────
print("🔧 Installing torchcrepe...")
run("uv pip install --system torchcrepe")

# ── 6. Core dependencies (py312-safe) ────────────────────────
print("🔧 Installing core dependencies...")
run("pip install -q "
    "numpy==1.26.4 "
    "scipy "
    "librosa==0.10.2 "
    "soundfile "
    "praat-parselmouth "
    "pyworld "
    "faiss-cpu "
    "scikit-learn "
    "einops "
    "tensorboard "
    "pydub "
    "ffmpeg-python "
    "gradio==3.50.2 "
    "colorama "
    "aiofiles "
    "edge-tts")

# ── 7. fairseq ───────────────────────────────────────────────
print("🔧 Installing fairseq...")
try:
    run("pip install -q fairseq")
    print("  fairseq (official) installed.")
except RuntimeError:
    print("  official fairseq failed, trying py312 fork...")
    run("pip install -q git+https://github.com/One-sixth/fairseq.git")
    print("  fairseq (fork) installed.")

# ── 8. Patch fairseq weights_only ────────────────────────────
print("🩹 Patching fairseq...")
try:
    import importlib, fairseq
    importlib.reload(fairseq)
    ckpt_path = os.path.join(os.path.dirname(fairseq.__file__), 'checkpoint_utils.py')
    with open(ckpt_path, 'r') as f:
        content = f.read()
    if 'weights_only=False' not in content:
        patched = content.replace(
            'state = torch.load(f, map_location=torch.device("cpu"))',
            'state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'
        )
        with open(ckpt_path, 'w') as f:
            f.write(patched)
        print("  patched.")
    else:
        print("  already patched.")
except Exception as e:
    print(f"  patch skipped: {e}")

# ── 9. Download pretrained weights ───────────────────────────
print("📥 Downloading pretrained weights...")
os.makedirs('/content/RVC/assets/pretrained_v2', exist_ok=True)
os.makedirs('/content/RVC/assets/hubert', exist_ok=True)

BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
weights = [
    (f"{BASE}/pretrained_v2/f0Ov2Super32kG.pth", "/content/RVC/assets/pretrained_v2/f0Ov2Super32kG.pth"),
    (f"{BASE}/pretrained_v2/f0Ov2Super32kD.pth", "/content/RVC/assets/pretrained_v2/f0Ov2Super32kD.pth"),
    (f"{BASE}/pretrained_v2/f0G32k.pth",         "/content/RVC/assets/pretrained_v2/f0G32k.pth"),
    (f"{BASE}/pretrained_v2/f0D32k.pth",         "/content/RVC/assets/pretrained_v2/f0D32k.pth"),
    (f"{BASE}/hubert_base.pt",                   "/content/RVC/assets/hubert/hubert_base.pt"),
    (f"{BASE}/rmvpe.pt",                         "/content/RVC/assets/rmvpe.pt"),
]
for url, dest in weights:
    if not os.path.exists(dest):
        print(f"  downloading {os.path.basename(dest)}...")
        os.system(f"wget -q --show-progress -O '{dest}' '{url}'")
    else:
        print(f"  {os.path.basename(dest)} already exists")

from IPython.display import clear_output, display
from ipywidgets import Button
clear_output()
print("✅ RVC fully installed & weights downloaded!")
display(Button(description="Done", button_style="success"))

---
## 🎓 Training

In [ ]:
#@title 🎵 **Step 2 : Upload Dataset (Vocals)**
#@markdown Upload one clean vocal audio file. Recommended: **3-7 minutes**, no background music or reverb.
#@markdown If you already uploaded and the runtime restarted, your file will be detected automatically.

import os, librosa, soundfile as sf
from IPython.display import Audio, display

os.makedirs('/content/dataset', exist_ok=True)
output_path = '/content/dataset/vocal_audio.wav'

if os.path.exists(output_path):
    print("✅ Audio already exists, skipping upload.")
    audio, sr = librosa.load(output_path, sr=None)
    duration = len(audio) / sr
    print(f"Duration: {duration/60:.2f} min | Sample rate: {sr} Hz")
    display(Audio(output_path))
else:
    print("📤 Please upload your vocal audio file...")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        fn = list(uploaded.keys())[0]
        audio, sr = librosa.load(fn, sr=None)
        sf.write(output_path, audio, sr, format='wav')
        if os.path.exists(fn) and fn != output_path:
            os.remove(fn)
        duration = len(audio) / sr
        print(f"✅ Saved: {output_path}")
        print(f"Duration: {duration/60:.2f} min | Sample rate: {sr} Hz")
        display(Audio(output_path))
    else:
        print("No file uploaded.")

In [ ]:
#@title ⚡ **Step 3 : Create Training Files**
#@markdown Enter a model name (English only, no spaces or symbols).
#@markdown Use the **exact same name** in Steps 3 and 4.

import os, shutil, traceback
import numpy as np
import faiss
from sklearn.cluster import MiniBatchKMeans
from pydub import AudioSegment
from IPython.display import clear_output, display
from ipywidgets import Button

# Safety checks
if not os.path.exists('/content/RVC'):
    raise RuntimeError("RVC not found! Run Step 1 first.")
if not os.path.exists('/content/dataset/vocal_audio.wav'):
    raise RuntimeError("No audio found! Run Step 2 first.")

os.chdir('/content/RVC')
now_dir = '/content/RVC'

model_name = '' #@param {type:"string"}
dataset_folder = '/content/dataset'

if not model_name.strip():
    raise ValueError("Model name cannot be empty!")

# ── Resume training support ──────────────────────────────────
logs_path = f'{now_dir}/logs/{model_name}'
temp_DG_path = '/content/temp_DG'

if os.path.exists(logs_path):
    print("Existing model found - resuming training.")
    os.makedirs(temp_DG_path, exist_ok=True)
    for item in os.listdir(logs_path):
        item_path = os.path.join(logs_path, item)
        if os.path.isfile(item_path) and (item.startswith('D_') or item.startswith('G_')) and item.endswith('.pth'):
            shutil.copy(item_path, temp_DG_path)
    for item in os.listdir(logs_path):
        item_path = os.path.join(logs_path, item)
        if os.path.isfile(item_path):
            os.remove(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
    for file_name in os.listdir(temp_DG_path):
        shutil.move(os.path.join(temp_DG_path, file_name), logs_path)
    shutil.rmtree(temp_DG_path)

# ── Dataset cache check ──────────────────────────────────────
try:
    duration = len(AudioSegment.from_file('/content/dataset/vocal_audio.wav')) / 1000.0
    cache = duration < 600
    print(f"Dataset duration: {duration/60:.2f} min | Cache in GPU: {cache}")
except:
    cache = False

# ── Preprocess ───────────────────────────────────────────────
os.makedirs(f'{now_dir}/logs/{model_name}', exist_ok=True)
with open(f'{now_dir}/logs/{model_name}/preprocess.log', 'w') as f:
    f.write('Starting...')

print("Processing audio...")
os.system(f"python {now_dir}/infer/modules/train/preprocess.py "
          f"{dataset_folder} 32000 2 {now_dir}/logs/{model_name} False 3.0 > /dev/null 2>&1")

with open(f'{now_dir}/logs/{model_name}/preprocess.log', 'r') as f:
    log_content = f.read()
if 'end preprocess' in log_content:
    print("Preprocessing complete.")
else:
    raise RuntimeError(f"Preprocessing failed. Log: {log_content[-500:]}")

# ── Extract F0 ───────────────────────────────────────────────
with open(f'{now_dir}/logs/{model_name}/extract_f0_feature.log', 'w') as f:
    f.write('Starting...')

print("Extracting pitch (F0)...")
os.system(f"python {now_dir}/infer/modules/train/extract/extract_f0_rmvpe.py "
          f"1 0 0 {now_dir}/logs/{model_name} True")

print("Extracting features...")
os.system(f"python {now_dir}/infer/modules/train/extract_feature_print.py "
          f"cuda:0 1 0 {now_dir}/logs/{model_name} v2 True")

with open(f'{now_dir}/logs/{model_name}/extract_f0_feature.log', 'r') as f:
    feat_log = f.read()
if 'all-feature-done' in feat_log:
    print("Feature extraction complete.")
else:
    print(f"Warning: feature log: {feat_log[-300:]}")

# ── Build FAISS Index ────────────────────────────────────────
print("Building FAISS index...")

def train_index(exp_dir1, version19):
    exp_dir = f"{now_dir}/logs/{exp_dir1}"
    feature_dir = f"{exp_dir}/3_feature768" if version19 == "v2" else f"{exp_dir}/3_feature256"
    if not os.path.exists(feature_dir) or len(os.listdir(feature_dir)) == 0:
        yield "Feature directory empty - extraction may have failed."
        return
    npys = [np.load(os.path.join(feature_dir, name)) for name in sorted(os.listdir(feature_dir))]
    big_npy = np.concatenate(npys, 0)
    np.random.shuffle(big_npy)
    if big_npy.shape[0] > 2e5:
        yield f"Running k-means on {big_npy.shape[0]} samples..."
        try:
            big_npy = MiniBatchKMeans(
                n_clusters=10000, verbose=True,
                batch_size=256 * os.cpu_count(),
                compute_labels=False, init="random"
            ).fit(big_npy).cluster_centers_
        except:
            yield traceback.format_exc()
    np.save(f"{exp_dir}/total_fea.npy", big_npy)
    n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
    index = faiss.index_factory(768 if version19 == "v2" else 256, f"IVF{n_ivf},Flat")
    faiss.extract_index_ivf(index).nprobe = 1
    index.train(big_npy)
    faiss.write_index(index, f"{exp_dir}/trained_IVF{n_ivf}_Flat_nprobe_{exp_dir1}_{version19}.index")
    yield "Adding vectors..."
    for i in range(0, big_npy.shape[0], 8192):
        index.add(big_npy[i:i + 8192])
    faiss.write_index(index, f"{exp_dir}/added_IVF{n_ivf}_Flat_nprobe_{exp_dir1}_{version19}.index")
    yield f"Index built: added_IVF{n_ivf}_Flat_nprobe_{exp_dir1}_{version19}.index"

for line in train_index(model_name, 'v2'):
    print(" ", line)

clear_output()
print(f"All training files created for model: {model_name}")
display(Button(description="Done", button_style="success"))

In [ ]:
#@title 🚀 **Step 4 : Train Model**
#@markdown Enter the **same model name** as Step 3.

import os, json, pathlib
from random import shuffle
from subprocess import Popen, PIPE, STDOUT

# Fix: hardcode path instead of relying on %cd magic
os.chdir('/content/RVC')
now_dir = '/content/RVC'

model_name = '' #@param {type:"string"}
#@markdown **Epochs** — recommended 400-600 for 10 min dataset
epochs = 400 #@param {type:"slider", min:50, max:5000, step:10}
#@markdown **Save frequency** — save checkpoint every N epochs
save_frequency = 50 #@param {type:"slider", min:10, max:200, step:10}

if not model_name.strip():
    raise ValueError("Model name cannot be empty!")

# ── Pre-flight checks ────────────────────────────────────────
if not os.path.exists(now_dir):
    raise RuntimeError("RVC not found! Run Step 1 first.")

for folder in ['0_gt_wavs', '3_feature768', '2a_f0', '2b-f0nsf']:
    p = f"{now_dir}/logs/{model_name}/{folder}"
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing folder: {p}\nRun Step 3 first!")

# ── Config ───────────────────────────────────────────────────
sample_rate = '32k'
batch_size = 8
OV2 = True

G_file = f'assets/pretrained_v2/f0Ov2Super{sample_rate}G.pth' if OV2 else f'assets/pretrained_v2/f0G{sample_rate}.pth'
D_file = f'assets/pretrained_v2/f0Ov2Super{sample_rate}D.pth' if OV2 else f'assets/pretrained_v2/f0D{sample_rate}.pth'

for f in [G_file, D_file]:
    if not os.path.exists(f"{now_dir}/{f}"):
        raise FileNotFoundError(f"Pretrained weight missing: {f}\nRe-run Step 1!")

print(f"Model      : {model_name}")
print(f"Epochs     : {epochs}")
print(f"Batch size : {batch_size}")
print(f"Pretrained : OV2 Super 32k")
print(f"Working dir: {now_dir}")

# ── Cache flag ───────────────────────────────────────────────
if 'cache' not in dir():
    try:
        from pydub import AudioSegment
        duration = len(AudioSegment.from_file('/content/dataset/vocal_audio.wav')) / 1000.0
        cache = duration < 600
    except:
        cache = False

def click_train(exp_dir1, sr2, if_f0_3, spk_id5, save_epoch10, total_epoch11,
                batch_size12, if_save_latest13, pretrained_G14, pretrained_D15,
                gpus16, if_cache_gpu17, if_save_every_weights18, version19):

    exp_dir     = f"{now_dir}/logs/{exp_dir1}"
    gt_wavs_dir = f"{exp_dir}/0_gt_wavs"
    feature_dir = f"{exp_dir}/3_feature768" if version19 == "v2" else f"{exp_dir}/3_feature256"
    f0_dir      = f"{exp_dir}/2a_f0"
    f0nsf_dir   = f"{exp_dir}/2b-f0nsf"

    names = (
        set([n.split(".")[0] for n in os.listdir(gt_wavs_dir)])
        & set([n.split(".")[0] for n in os.listdir(feature_dir)])
        & set([n.split(".")[0] for n in os.listdir(f0_dir)])
        & set([n.split(".")[0] for n in os.listdir(f0nsf_dir)])
    ) if if_f0_3 else (
        set([n.split(".")[0] for n in os.listdir(gt_wavs_dir)])
        & set([n.split(".")[0] for n in os.listdir(feature_dir)])
    )

    print(f"Training samples found: {len(names)}")
    if len(names) == 0:
        raise ValueError("No matching training samples found! Re-run Step 3.")

    opt = []
    for name in names:
        if if_f0_3:
            opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{f0_dir}/{name}.wav.npy|{f0nsf_dir}/{name}.wav.npy|{spk_id5}")
        else:
            opt.append(f"{gt_wavs_dir}/{name}.wav|{feature_dir}/{name}.npy|{spk_id5}")

    fea_dim = 256 if version19 == "v1" else 768
    for _ in range(2):
        if if_f0_3:
            opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{sr2}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|{spk_id5}")
        else:
            opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{sr2}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{spk_id5}")

    shuffle(opt)
    with open(f"{exp_dir}/filelist.txt", "w") as f:
        f.write("\n".join(opt))
    print(f"filelist.txt written ({len(opt)} entries)")

    config_path = f"configs/v2/{sr2}.json" if version19 == "v2" and sr2 != "40k" else f"configs/v1/{sr2}.json"
    config_save_path = os.path.join(exp_dir, "config.json")
    if not pathlib.Path(config_save_path).exists():
        with open(config_save_path, "w", encoding="utf-8") as f:
            with open(config_path, "r") as cfg:
                json.dump(json.load(cfg), f, ensure_ascii=False, indent=4, sort_keys=True)
            f.write("\n")

    cmd = (
        f'python infer/modules/train/train.py -e "{exp_dir1}" -sr {sr2} -f0 {1 if if_f0_3 else 0} '
        f'-bs {batch_size12} -g {gpus16} -te {total_epoch11} -se {save_epoch10} '
        f'{"-pg " + pretrained_G14 if pretrained_G14 else ""} '
        f'{"-pd " + pretrained_D15 if pretrained_D15 else ""} '
        f'-l {1 if if_save_latest13 else 0} -c {1 if if_cache_gpu17 else 0} '
        f'-sw {1 if if_save_every_weights18 else 0} -v {version19}'
    )
    print(f"\nStarting training - {total_epoch11} epochs...\n")

    p = Popen(cmd, shell=True, cwd=now_dir, stdout=PIPE, stderr=STDOUT, bufsize=1, universal_newlines=True)
    for line in p.stdout:
        print(line.strip())
    p.wait()
    return "Training complete!"

%load_ext tensorboard
%tensorboard --logdir /content/RVC/logs --port=8888

result = click_train(
    model_name, sample_rate, True, 0,
    save_frequency, epochs, batch_size,
    True, G_file, D_file, 0, cache, True, 'v2'
)
print(result)

---
## 💾 Backup & Load Models

In [ ]:
#@title 📤 **Backup Model to Google Drive**
#@markdown Saves your model to `RVC_Models` folder on Google Drive.

import os, tarfile
from google.colab import drive

Model_Name = '' #@param {type:"string"}

if not Model_Name.strip():
    raise ValueError("Model name cannot be empty!")

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/RVC_Models', exist_ok=True)

logs_src    = f'/content/RVC/logs/{Model_Name}'
weights_src = f'/content/RVC/assets/weights/{Model_Name}.pth'
out_path    = f'/content/drive/MyDrive/RVC_Models/{Model_Name}.tar.gz'

if not os.path.exists(logs_src):
    raise FileNotFoundError(f"No logs for model '{Model_Name}'. Train it first!")

print(f"Backing up {Model_Name}...")
with tarfile.open(out_path, 'w:gz') as tar:
    tar.add(logs_src, arcname=f'logs/{Model_Name}')
    if os.path.exists(weights_src):
        tar.add(weights_src, arcname=f'assets/weights/{Model_Name}.pth')
        print("  weights included.")

print(f"Backed up to: Google Drive/RVC_Models/{Model_Name}.tar.gz")

In [ ]:
#@title 📥 **Load Model from Google Drive**
#@markdown Your model `.tar.gz` must be in `RVC_Models` folder on Google Drive.

import os, tarfile
from google.colab import drive
from ipywidgets import Button
from IPython.display import display

Model_Name = '' #@param {type:"string"}

if not Model_Name.strip():
    raise ValueError("Model name cannot be empty!")

drive.mount('/content/drive')

drive_path = f"/content/drive/MyDrive/RVC_Models/{Model_Name}.tar.gz"
local_copy = f"/content/{Model_Name}.tar.gz"

if not os.path.exists(drive_path):
    raise FileNotFoundError(f"Not found: Google Drive/RVC_Models/{Model_Name}.tar.gz")

print("Copying from Drive...")
os.system(f'cp "{drive_path}" "{local_copy}"')

print("Extracting...")
os.makedirs('/content/RVC', exist_ok=True)
with tarfile.open(local_copy, 'r:gz') as tar:
    tar.extractall('/content/RVC')
os.remove(local_copy)

if os.path.exists('/content/RVC/weights'):
    os.system("cp -R /content/RVC/weights /content/RVC/assets")
    os.system("rm -R /content/RVC/weights")

print(f"Model '{Model_Name}' loaded!")
display(Button(description="Done", button_style="success"))

In [ ]:
#@title ⬇️ **Download .pth File**

import os
from google.colab import files

Model_Name = '' #@param {type:"string"}
file_path = f'/content/RVC/assets/weights/{Model_Name}.pth'

if os.path.exists(file_path):
    print(f"Downloading: {Model_Name}.pth")
    files.download(file_path)
else:
    print(f"File not found: {file_path}")
    print("Check model name is correct and training completed.")

In [ ]:
#@title ⬇️ **Download .index File**

import os
from google.colab import files

Model_Name = '' #@param {type:"string"}
index_dir = f'/content/RVC/logs/{Model_Name}'

if not os.path.exists(index_dir):
    print(f"Logs folder not found for: {Model_Name}")
else:
    target = None
    for fname in os.listdir(index_dir):
        if fname.startswith('added') and fname.endswith('.index'):
            target = os.path.join(index_dir, fname)
            break
    if target:
        print(f"Downloading: {os.path.basename(target)}")
        files.download(target)
    else:
        print("No .index file found. Run Step 3 first.")

---
## 🔊 Inference — Voice Conversion

In [ ]:
#@title 🎤 **Inference Step 1 : Upload Target Audio**
#@markdown Upload the audio you want to convert into your cloned voice.
#@markdown Already uploaded files will be detected automatically.

import os, librosa, soundfile as sf
from IPython.display import Audio, display

os.makedirs('/content/sample_data', exist_ok=True)
audio_path = '/content/sample_data/input_audio.wav'

if os.path.exists(audio_path):
    print("Audio already exists, skipping upload.")
    display(Audio(audio_path))
else:
    print("Upload your target audio file...")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        fn = list(uploaded.keys())[0]
        audio, sr = librosa.load(fn, sr=None)
        sf.write(audio_path, audio, sr, format='wav')
        if os.path.exists(fn) and fn != audio_path:
            os.remove(fn)
        duration = len(audio) / sr
        print(f"Saved: {audio_path} | {duration:.1f}s | {sr} Hz")
        display(Audio(audio_path))
    else:
        print("No file uploaded.")

In [ ]:
#@title 🔁 **Inference Step 2 : Run Voice Conversion**

import os
import IPython.display as ipd

os.chdir('/content/RVC')
now_dir = '/content/RVC'

model_name = '' #@param {type:"string"}
#@markdown **Pitch shift** — same gender: 0 | Male to Female: +12 | Female to Male: -12
pitch = 0 #@param {type:"slider", min:-24, max:24, step:1}

if not model_name.strip():
    raise ValueError("Model name cannot be empty!")

# ── Locate files ─────────────────────────────────────────────
model_path = f"{now_dir}/assets/weights/{model_name}.pth"
index_dir  = f"{now_dir}/logs/{model_name}"
input_path = "/content/sample_data/input_audio.wav"
save_as    = f"{now_dir}/audios/output_audio.wav"

os.makedirs(f"{now_dir}/audios", exist_ok=True)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")
if not os.path.exists(input_path):
    raise FileNotFoundError("Input audio not found. Run Inference Step 1 first.")

# Auto-find index file
index_filename = None
if os.path.exists(index_dir):
    for fname in os.listdir(index_dir):
        if fname.startswith('added') and fname.endswith('.index'):
            index_filename = fname
            break

if not index_filename:
    raise FileNotFoundError(f"No .index file found in {index_dir}. Run Step 3 first.")

index_path = f"{index_dir}/{index_filename}"

print(f"Model  : {model_name}")
print(f"Index  : {index_filename}")
print(f"Pitch  : {pitch:+d} semitones")

os.environ['weight_root'] = os.path.dirname(model_path)
os.environ['index_root']  = os.path.dirname(index_path)

model_filename = os.path.basename(model_path)
index_basename = os.path.basename(index_path)

os.system(f"rm -f '{save_as}'")

os.system(
    f"python {now_dir}/tools/cmd/infer_cli.py "
    f"--f0up_key {pitch} "
    f"--input_path '{input_path}' "
    f"--index_path '{index_basename}' "
    f"--f0method rmvpe "
    f"--opt_path '{save_as}' "
    f"--model_name '{model_filename}' "
    f"--index_rate 0.5 "
    f"--device cuda:0 "
    f"--is_half True "
    f"--filter_radius 3 "
    f"--resample_sr 0 "
    f"--rms_mix_rate 0 "
    f"--protect 0.5"
)

if os.path.exists(save_as):
    print("\nConversion complete!")
    ipd.display(ipd.Audio(save_as))
else:
    print("\nOutput not generated. Check errors above.")

In [ ]:
#@title 👂 **Preview Output Audio**

from IPython.display import Audio, display
import os

audio_path = '/content/RVC/audios/output_audio.wav'
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print(f"Output ready ({size_mb:.2f} MB)")
    display(Audio(audio_path))
else:
    print("No output audio. Run Inference Step 2 first.")

In [ ]:
#@title ⬇️ **Download Output Audio**
#@markdown Downloads the converted audio. Works even for files longer than 4 minutes.

from google.colab import files
import os

file_path = '/content/RVC/audios/output_audio.wav'
if os.path.exists(file_path):
    print("Downloading output audio...")
    files.download(file_path)
else:
    print("No output audio found. Run inference first.")

---
<div style="text-align:center; color:#666; font-size:0.85em; padding:16px;">
  Made with love by <strong>Yogeshwar AI</strong> &nbsp;·&nbsp;
  <a href='https://www.instagram.com/theyogeshwar' style='color:#a78bfa;'>@theyogeshwar</a><br>
  Use responsibly. Only clone voices you have permission for.
</div>